# Issue #10 — run LLaMA 3.2 / Mistral 7B on the **segmented** condition (GPU)

Dedicated notebook for issue #10. Same pipeline as issue #9, with
`CONDITION = 'segmented'` preset (morphologically segmented input from YAP).

Fully resumable: results + response cache live on Drive — after any disconnect,
rerun cells 1–4 and then cell 6.

Runtime: **T4 GPU**. Smoke test first (`SMOKE_LIMIT = 20`), then full run (~4–5h).

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive (persistent cache + results)
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST = '/content/drive/MyDrive/nlp_final_runs'
os.makedirs(PERSIST, exist_ok=True)

In [ ]:
# 3. Clone the repo (private -> needs PAT) and install deps
from getpass import getpass
import os, subprocess
if not os.path.exists('/content/NLP-Final-'):
    pat = getpass('GitHub PAT: ')
    subprocess.run(['git', 'clone',
        f'https://{pat}@github.com/AdonZahavi/NLP-Final-.git',
        '/content/NLP-Final-'], check=True)
%cd /content/NLP-Final-
!git pull
!pip -q install transformers accelerate sentencepiece python-dotenv
!pip -q install -U datasets

In [ ]:
# 4. HF token + link Drive-backed cache & results
from huggingface_hub import login
login()  # HF token (may not prompt if already stored)

import os
for name in ('cache', 'results'):
    target = f'{PERSIST}/{name}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(name):
        os.unlink(name)
    elif os.path.isdir(name):
        import shutil; shutil.rmtree(name)
    os.symlink(target, name)
print('cache ->', os.path.realpath('cache'))
print('results ->', os.path.realpath('results'))
# clean any zero-byte leftovers from crashed runs
!find -L results -name "*.jsonl" -size -2c -print -delete

In [ ]:
# 5. Configuration — issue #10 = segmented
CONDITION = 'segmented'
MODELS = ['llama-3.2', 'mistral-7b']   # LLaMA first (smaller)
SMOKE_LIMIT = 20    # 20 = smoke test; set 0 for the full run

In [ ]:
# 6. Smoke + full run (resumable — rerun this cell after any crash)
import subprocess, sys, os
for model in MODELS:
    cmd = [sys.executable, 'scripts/run_all.py',
           '--condition', CONDITION, '--models', model]
    if SMOKE_LIMIT:
        cmd += ['--limit', str(SMOKE_LIMIT)]
    print('\n=== ', ' '.join(cmd), ' ===')
    env = {**os.environ, 'PYTHONPATH': 'src',
           'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
    r = subprocess.run(cmd, env=env)
    if r.returncode != 0:
        print(f'!! {model} exited {r.returncode} — rerun this cell to resume')

In [ ]:
# 7. Sanity report
!PYTHONPATH=src python scripts/summarize_results.py --condition {CONDITION}

In [ ]:
# 8. Copy results into the repo tree and push
import shutil, os, subprocess
subprocess.run(['git', 'config', 'user.email', 'orna.zahavi1@gmail.com'], check=True)
subprocess.run(['git', 'config', 'user.name', 'AdonZahavi'], check=True)
if os.path.islink('results'):
    os.unlink('results')
    shutil.copytree(f'{PERSIST}/results', 'results')
!git add results/ && git status --short results/ | head
BRANCH = f'issue-10-hf-runs-{CONDITION}'
!git checkout -b {BRANCH} 2>/dev/null || git checkout {BRANCH}
!git commit -m "HF model results: {CONDITION} condition (llama-3.2, mistral-7b)"
!git push -u origin {BRANCH}